# 02 Train, validation and test split

Goal: divide the images into a training, validation and test set in a way that limits leakage between them.

- **Train** (70%): the model learns from these images.
- **Validation** (15%): used to make choices during development, such as the number of epochs.
- **Test** (15%): used once, at the end, to measure final performance.

In [42]:
import numpy as np
import csv
from PIL import Image
from pathlib import Path

data_dir = Path("../data/Original CT Scans")

def file_number(path):
    return int(path.stem.replace(path.parent.name, ""))

def has_black_corners(path):
    img = np.array(Image.open(path).convert("L"))
    corner = img[:10, :10]
    return corner.mean() < 10
    
def block_split(files):
    n = len(files)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    train = files[:n_train]
    val = files[n_train:n_train + n_val]
    test = files[n_train + n_val:]
    return train, val, test

## 1. Sorting files by number

`glob` returns files in alphabetical order (`nCT1`, `nCT10`, `nCT100`). The block split needs the real order, so `file_number` extracts the number from each file name.

In [30]:
print(file_number(data_dir / "nCT" / "nCT10.jpg"))

10


In [31]:
for class_dir in data_dir.iterdir():
    files = list(class_dir.glob("*"))
    files = sorted(files, key=file_number)  # sort by file number
    print(class_dir.name)
    for f in files[:5]:
        print(f.name)

nCT
nCT1.jpg
nCT2.jpg
nCT3.jpg
nCT4.jpg
nCT5.jpg
NiCT
NiCT1.jpg
NiCT2.jpg
NiCT3.jpg
NiCT4.jpg
NiCT5.jpg
pCT
pCT1.jpg
pCT2.jpg
pCT3.jpg
pCT4.jpg
pCT5.jpg


## 2. First attempt: block split

The original project splits at random. In notebook 01, I saw that neighbouring file numbers often look very similar, which suggests they come from the same scan. With a random split, near-identical slices end up in both train and test, which inflates the test score.

Instead, I sort each class by file number and assign the first 70% to train, the next 15% to validation and the last 15% to test. Neighbouring files stay together, except at the two boundaries.

Assumption: file numbering roughly follows the scans. My sample in notebook 01 supports this, but it cannot be proven without patient IDs.

In [32]:
train_files, val_files, test_files = [], [], []

for class_dir in data_dir.iterdir():
    files = sorted(class_dir.glob("*"), key=file_number)
    n = len(files)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    train = files[:n_train]
    val = files[n_train:n_train + n_val]   # the next n_val files after train
    test = files[n_train + n_val:]  # everything that is left

    print(class_dir.name, len(train), len(val), len(test))

    train_files.extend(train)
    val_files.extend(val)
    test_files.extend(test)

print("Total", len(train_files), len(val_files), len(test_files))

nCT 6985 1496 1498
NiCT 3993 855 857
pCT 2800 600 601
Total 13778 2951 2956


In [33]:
for split_name, split_files in zip(["train", "val", "test"], [train_files, val_files, test_files]):
    for class_name in ["nCT", "NiCT", "pCT"]:
        total = 0
        count = 0
        for f in split_files:
            if f.parent.name == class_name:      # does f belong to class_name?
                total += 1
                if has_black_corners(f):  # does f have black corners?
                    count += 1
        print(split_name, class_name, count, total, f"{count / total * 100:.1f}%")

train nCT 1610 6985 23.0%
train NiCT 833 3993 20.9%
train pCT 12 2800 0.4%
val nCT 224 1496 15.0%
val NiCT 0 855 0.0%
val pCT 0 600 0.0%
test nCT 15 1498 1.0%
test NiCT 0 857 0.0%
test pCT 1 601 0.2%


**Findings**

The split has no overlap between sets, but the share of images with black corners differs strongly:

- `nCT`: 23.0% in train, 15.0% in validation, 1.0% in test
- `NiCT`: 20.9% in train, 0.0% in validation and test
- `pCT`: 0.4% in train, 0.0% in validation, 0.2% in test

Images with black corners almost all have low file numbers. This confirms that the numbering follows how the data was collected, but it also means the three sets contain different types of images. The validation set would then be a poor predictor of test performance.

## 3. Block split per image type

To fix this, I first divide each class into two groups: images with and without black corners. I apply the same 70/15/15 block split to each group and then combine the results. Slices from the same scan share the same image type, so neighbours still stay together, and every set gets the same mix of image types.

In [37]:
train_files, val_files, test_files = [], [], []

for class_dir in data_dir.iterdir():
    files = sorted(class_dir.glob("*"), key=file_number)

    corner_files = []
    square_files = []
    for f in files:
        if has_black_corners(f):  # does f have black corners?
            corner_files.append(f)
        else:
            square_files.append(f)

    for group in [corner_files, square_files]:
        train, val, test = block_split(group)
        train_files.extend(train)
        val_files.extend(val)
        test_files.extend(test)

print("Total", len(train_files), len(val_files), len(test_files))

Total 13778 2949 2958


In [38]:
for split_name, split_files in zip(["train", "val", "test"], [train_files, val_files, test_files]):
    for class_name in ["nCT", "NiCT", "pCT"]:
        total = 0
        count = 0
        for f in split_files:
            if f.parent.name == class_name:      # does f belong to class_name?
                total += 1
                if has_black_corners(f):  # does f have black corners?
                    count += 1
        print(split_name, class_name, count, total, f"{count / total * 100:.1f}%")

train nCT 1294 6985 18.5%
train NiCT 583 3993 14.6%
train pCT 9 2800 0.3%
val nCT 277 1496 18.5%
val NiCT 124 854 14.5%
val pCT 1 599 0.2%
test nCT 278 1498 18.6%
test NiCT 126 858 14.7%
test pCT 3 602 0.5%


**Findings**

- No image appears in more than one set.
- Set sizes: 13,778 train, 2,949 validation, 2,958 test.
- The share of images with black corners is now almost equal across sets: about 18.5% for `nCT`, 14.6% for `NiCT` and 0.2% to 0.5% for `pCT`. The small variation for `pCT` comes from there being only 13 such images in total.

## 4. Saving the split

The split is saved to `splits/block_split.csv`, with one row per image: its path relative to the data folder, its label and its set. Later notebooks read this file, so every experiment uses exactly the same split. The file is part of the repository, so anyone can reproduce it.

In [43]:
split_dir = Path("../splits")
split_dir.mkdir(exist_ok=True)

with open(split_dir / "block_split.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["path", "label", "split"])
    for split_name, split_files in zip(["train", "val", "test"], [train_files, val_files, test_files]):
        for f in split_files:
            writer.writerow([f.relative_to(data_dir).as_posix(), f.parent.name, split_name])

## Summary and limitations

- The split keeps neighbouring files together, which reduces, but does not remove, leakage between sets. Without patient IDs, slices from one patient can still end up in two sets at the block boundaries, or when a patient's slices are not numbered consecutively.
- Each set has the same mix of image types, so validation results should be a fair predictor of test results.
- Next step: train a model on this split. Later, I can compare it with a random split to see how much leakage affects the test score.

In [41]:
import random

random.seed(42)
random_train, random_val, random_test = [], [], []

for class_dir in data_dir.iterdir():
    files = sorted(class_dir.glob("*"), key=file_number)
    random.shuffle(files)
    train, val, test = block_split(files)
    random_train.extend(train)
    random_val.extend(val)
    random_test.extend(test)

print(len(random_train), len(random_val), len(random_test))

13778 2951 2956


In [44]:
with open(split_dir / "random_split.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["path", "label", "split"])
    for split_name, split_files in zip(["train", "val", "test"], [random_train, random_val, random_test]):
        for f in split_files:
            writer.writerow([f.relative_to(data_dir).as_posix(), f.parent.name, split_name])